In [1]:
print('ok')

ok


In [ ]:
# ffmpeg -i input_video.mp4 -vcodec libx264 -crf 25 \
#        -x264opts "bframes=0:keyint=30" encoded_output_video.mp4

In [2]:
!pip install requests

In [6]:
import requests
spec = requests.get("http://localhost:8000/openapi.json", timeout=10).json()
for p in sorted(spec["paths"]):
    print(p, list(spec["paths"][p].keys()))

/api/v1/attribute_search ['post']
/api/v1/attribute_search/atif ['post']
/api/v1/attribute_search/full ['post']
/api/v1/attribute_search/stream ['post']
/api/v1/critic ['post']
/api/v1/critic/atif ['post']
/api/v1/critic/full ['post']
/api/v1/critic/stream ['post']
/api/v1/embed_search ['post']
/api/v1/embed_search/atif ['post']
/api/v1/embed_search/full ['post']
/api/v1/embed_search/stream ['post']
/api/v1/rtsp-streams/add ['post']
/api/v1/rtsp-streams/delete/{name} ['delete']
/api/v1/search ['post']
/api/v1/search/atif ['post']
/api/v1/search/full ['post']
/api/v1/search/stream ['post']
/api/v1/videos ['post']
/api/v1/videos-for-search/{filename} ['put']
/api/v1/videos/{sensor_id}/complete ['post']
/api/v1/videos/{video_id} ['delete']
/auth/redirect ['get']
/chat ['post']
/chat/stream ['post']
/evaluate ['post']
/evaluate/item ['post']
/evaluate/job/last ['get']
/evaluate/job/{job_id} ['get']
/evaluate/jobs ['get']
/executions/{execution_id} ['get']
/executions/{execution_id}/interac

In [ ]:
curl -s "http://localhost:30888/api/v1/sensor/list" | jq .

In [ ]:
# import requests
# VSS = "http://localhost:8000/api/v1"   # backend port, NOT the UI port
# r = requests.post(f"{VSS}/files", data={
#     "filename": "/opt/nvidia/via/streams/warehouse.mp4",
#     "purpose": "vision",
#     "media_type": "video",
# })
# file_id = r.json()["id"]

KeyError: 'id'

In [ ]:
import os
from pathlib import Path
 
import requests
 
 
def query_vss_video(video_path: str, prompt: str) -> str:
    base_url = os.environ.get(
        "VSS_RTVLM_URL",
        "http://127.0.0.1:8018",
    ).rstrip("/") + "/v1"
 
    path = Path(video_path)
    if not path.is_file():
        raise FileNotFoundError(f"Video does not exist: {path}")
 
    # Confirm that RT-VLM is running.
    ready = requests.get(f"{base_url}/ready", timeout=15)
    if not ready.ok:
        raise RuntimeError(
            f"RT-VLM is not ready ({ready.status_code}): {ready.text[:1000]}"
        )
 
    # Get the precise model identifier loaded in this service.
    models = requests.get(f"{base_url}/models", timeout=15)
    if not models.ok:
        raise RuntimeError(
            f"Could not retrieve RT-VLM models ({models.status_code}): "
            f"{models.text[:1000]}"
        )
 
    model_data = models.json().get("data", [])
    if not model_data:
        raise RuntimeError(f"RT-VLM returned no models: {models.json()}")
 
    model_id = model_data[0]["id"]
 
    # Upload video.
    with path.open("rb") as video_file:
        uploaded = requests.post(
            f"{base_url}/files",
            files={
                "file": (
                    path.name,
                    video_file,
                    "video/mp4",
                )
            },
            data={
                "purpose": "vision",
                "media_type": "video",
            },
            timeout=(15, 300),
        )
 
    if not uploaded.ok:
        raise RuntimeError(
            f"RT-VLM video upload failed ({uploaded.status_code}): "
            f"{uploaded.text[:2000]}"
        )
 
    upload_data = uploaded.json()
    file_id = upload_data.get("id")
    if not file_id:
        raise RuntimeError(f"Upload response has no file ID: {upload_data}")
 
    try:
        # Analyze the complete video in timestamped chunks.
        captions = requests.post(
            f"{base_url}/generate_captions",
            json={
                "id": file_id,
                "prompt": prompt,
                "model": model_id,
                "chunk_duration": 60,
                "chunk_overlap_duration": 10,
                "enable_audio": False,
            },
            timeout=(15, 600),
        )
 
        if not captions.ok:
            raise RuntimeError(
                f"RT-VLM captioning failed ({captions.status_code}): "
                f"{captions.text[:2000]}"
            )
 
        result = captions.json()
        chunks = result.get("chunk_responses", [])
 
        if not chunks:
            raise RuntimeError(f"RT-VLM returned no caption chunks: {result}")
 
        return "\n\n".join(
            f"[{chunk.get('start_time', '?')} – {chunk.get('end_time', '?')}]\n"
            f"{chunk.get('content', '')}"
            for chunk in chunks
        ).strip()
 
    finally:
        # Remove VSS's temporary stored copy.
        try:
            requests.delete(f"{base_url}/files/{file_id}", timeout=30)
        except requests.RequestException:
            pass

'[0.0 – 2.634]\nblue'

In [ ]:
import random

options = [
    "raise your hands to the sky", 
    "do the ninja", 
    "scratch your scalp like a monkey", 
    "rub your stomach", 
    "hide your eyes", 
    "turn around", 
]

selected_action = random.choice(foo)

# TODO
# video = here

prompt = f"""
Assign a number to each person from left to right.
Who is the first person to follow the following action(s), in order:
{sequence}.
At what timestamp does this person execute the actions ?
"""

query_vss_video("cringe_video.mp4", prompt)

In [59]:
sequence = "- raise your hand"
prompt = f"""
Assign a number to each person from left to right.
Who is the first person to follow the following action(s), in order:
{sequence}.
At what timestamp does this person execute the actions ?
"""
query_vss_video("cringe_video.mp4", prompt)

'[0.0 – 13.52]\nThe person who is first to follow the action(s) is: Person 2, and the timestamp they execute it at is: 0.16.'

In [63]:
sequence = "- raise your hand"
prompt = f"""
Describe the outfit first person to follow the following action(s), in order:
{sequence}.
At what timestamp does this person execute the actions ?
"""
query_vss_video("cringe_video.mp4", prompt)

'[0.0 – 13.52]\nI am wearing a yellow t-shirt and a yellow bandana, with sunglasses perched on top of my head. I have a mustache and a tattoo on my left arm. I raise my right hand towards the camera, showing off my sleeveless black tank top and green earrings.'

In [53]:
import requests
response = requests.get("http://127.0.0.1:8000/v1/models")
model_id = response.json()["data"][0]["id"]
print(f"Loaded model: {model_id}")  # use this as the "model" field

KeyError: 'data'

In [28]:
sensor_id

'9c764615-c0dc-468d-b8ef-4c1227a421d0'

In [34]:
import requests, uuid, pathlib, json

HOST      = "http://localhost:8000"
FILE_PATH = pathlib.Path("encoded_video_2.mp4")
FILENAME  = FILE_PATH.name
s = requests.Session()

# 1. get the chunked-upload URL
r1 = s.post(f"{HOST}/api/v1/videos", json={"filename": FILENAME}, timeout=(10, 60))
r1.raise_for_status()                      # fail loudly instead of KeyError
upload_url = r1.json()["url"]
print("upload url:", upload_url)

# 2. push bytes to the VST/nvstreamer URL
with FILE_PATH.open("rb") as f:
    r2 = s.post(upload_url,
        headers={
            "nvstreamer-chunk-number":  "1",
            "nvstreamer-total-chunks":  "1",
            "nvstreamer-is-last-chunk": "true",
            "nvstreamer-identifier":    str(uuid.uuid4()),
            "nvstreamer-file-name":     FILENAME,
        },
        files={"mediaFile": (FILENAME, f, "video/mp4")},
        data={"filename": FILENAME,
              "metadata": json.dumps({"timestamp": "2025-01-01T00:00:00"})},
        timeout=(10, 1800))
r2.raise_for_status()
up = r2.json()
sensor_id = up.get("sensorId")
if not sensor_id:
    raise RuntimeError(f"no sensorId: {up}")

# 3. notify the agent -> fans out to RTVI-CV + RTVI-Embed
r3 = s.post(f"{HOST}/api/v1/videos/{sensor_id}/complete",
            json={**up, "filename": FILENAME}, timeout=(10, 1800))
r3.raise_for_status()
print("chunks_processed:", r3.json().get("chunks_processed"))   # must be > 0

# 4. query
print(s.post(f"{HOST}/generate",
             json={"input_message": "describe what happens in the video"},
             timeout=(10, 900)).json())

upload url: http://172.16.0.189:7777/vst/api/v1/storage/file


HTTPError: 502 Server Error: Bad Gateway for url: http://localhost:8000/api/v1/videos/9126f441-4936-4525-b9fb-e700a6af2881/complete

In [36]:
sensor_id

'9126f441-4936-4525-b9fb-e700a6af2881'

In [35]:
try:
    r3 = s.post(f"{HOST}/api/v1/videos/{sensor_id}/complete",
                json={**up, "filename": FILENAME}, timeout=(10, 1800))
    r3.raise_for_status()
except requests.HTTPError:
    print(r3.status_code, r3.text)   # <- the real reason lives here
    raise

502 {"detail":"RTVI-CV add failed: RTVI-CV returned 500: {\n\t\"reason\" : \"STREAM_ADD_FAIL, Duplicate Camera id, unable to add stream\",\n\t\"status\" : \"HTTP/1.1 500 Internal Server Error\"\n}"}


HTTPError: 502 Server Error: Bad Gateway for url: http://localhost:8000/api/v1/videos/9126f441-4936-4525-b9fb-e700a6af2881/complete

In [ ]:
HOST = "http://localhost:8000"
FILENAME = "encoded_output_video.mp4"

# --- 1. Upload the video -------------------------------------------------
with open(FILENAME, "rb") as f:
    r = requests.put(
        f"{HOST}/api/v1/videos-for-search/{FILENAME}",
        data=f,
        headers={"Content-Type": "video/mp4"},
        timeout=(10, 1800),
    )
print(r.status_code, r.text[:500])

502 {"detail":"RTVI-CV add failed: RTVI-CV returned 500: {\n\t\"reason\" : \"STREAM_ADD_FAIL, Duplicate Camera id, unable to add stream\",\n\t\"status\" : \"HTTP/1.1 500 Internal Server Error\"\n}"}


In [20]:
import requests, json

HOST = "http://localhost:8000"
VIDEO = "encoded_output_video.mp4"

# --- 1. Upload the file (multipart, NOT a raw PUT) ---------------------
with open(VIDEO, "rb") as f:
    r = requests.post(
        f"{HOST}/files",
        files={"file": (VIDEO, f, "video/mp4")},
        data={"purpose": "vision", "media_type": "video"},
        timeout=(10, 1800),
    )
r.raise_for_status()
file_id = r.json()["id"]
print("file_id:", file_id)

# --- 2. Summarize with YOUR prompt -------------------------------------
payload = {
    "id": file_id,
    "prompt": (
        "You are a monitoring assistant. Describe everything happening "
        "in the scene, including people, objects and any anomaly. "
        "Start each sentence with the timestamp."
    ),
    "caption_summarization_prompt":
        "Aggregate these per-chunk captions into a coherent timeline.",
    "summary_aggregation_prompt":
        "Produce a concise report with a bullet list of key events.",
    "chunk_duration": 30,           # seconds per VLM chunk
    "chunk_overlap_duration": 0,
    "max_tokens": 512,
    "temperature": 0.2,
    "top_p": 1,
    "top_k": 100,
    "enable_chat": True,            # required if you want Q&A afterwards
    "stream": False,
}

r = requests.post(f"{HOST}/summarize", json=payload, timeout=(10, 3600))
r.raise_for_status()
print(r.json()["choices"][0]["message"]["content"])

# --- 3. Ask follow-up questions ----------------------------------------
q = requests.post(f"{HOST}/chat/completions", json={
    "id": file_id,
    "messages": [{"role": "user", "content": "Was anyone without a helmet?"}],
}, timeout=(10, 600))
print(q.json()["choices"][0]["message"]["content"])

# --- 4. Clean up (important — frees GPU + keeps the graph DB small) -----
requests.delete(f"{HOST}/files/{file_id}")

HTTPError: 404 Client Error: Not Found for url: http://localhost:8000/files

In [18]:
import requests
HOST = "http://localhost:8000"
FILENAME = "encoded_output_video.mp4"

def purge(filename):
    vids = requests.get(f"{HOST}/api/v1/videos", timeout=30).json()
    items = vids if isinstance(vids, list) else vids.get("videos", vids.get("data", []))
    for v in items:
        name = v.get("filename") or v.get("name") or v.get("sensorName", "")
        if filename in name:
            vid = v.get("id") or v.get("sensorId") or v.get("streamId")
            print("deleting", vid, name)
            print(requests.delete(f"{HOST}/api/v1/videos/{vid}", timeout=120).status_code)

purge(FILENAME)   # always run before re-uploading the same name

In [16]:
import json

LVS = "http://localhost:8000"

payload = {
    "url": FILENAME,                     # from VIOS: /storage/file/<streamId>/url?container=mp4
    "model": "nim_nvidia_cosmos3-nano-reasoner_bf16-final",
    "scenario": "warehouse safety monitoring",
    "events": ["worker without hard hat", "forklift near-miss"],
    "chunk_duration": 10,
    "num_frames_per_second_or_fixed_frames_chunk": 20,
    "use_fps_for_chunking": False,
    "seed": 1,
}
r = requests.post(f"{LVS}/v1/summarize", json=payload, timeout=1800).json()
content = json.loads(r["choices"][0]["message"]["content"])
print(content["video_summary"])
print(content["events"])

KeyError: 'choices'

In [ ]:
# --- 2. Discover the VLM id ---------------------------------------------
model = requests.get(f"{VSS}/models").json()["data"][0]["id"]

In [ ]:
# --- 3. Summarize with YOUR prompt --------------------------------------
payload = {
    "id": file_id,
    "model": model,
    "chunk_duration": 10,          # seconds per VLM chunk
    "prompt": "You are a warehouse monitoring system. Describe the events "
              "and flag any safety anomaly. Timestamp every sentence.",
    "system_prompt": "You are a helpful assistant. Answer the user's question.",
    "caption_summarization_prompt":
        "Summarize sequential similar captions in the format "
        "start_time:end_time:caption, keeping all details.",
    "summary_aggregation_prompt":
        "Aggregate the captions into bullet points in the format "
        "start_time:end_time: detailed_event_description.",
    "enable_chat": True,           # needed if you want Q&A afterwards
    "stream": False,
}
resp = requests.post(f"{VSS}/summarize", json=payload, timeout=3600)
resp.raise_for_status()
print(resp.json()["choices"][0]["message"]["content"])

# --- 4. Optional follow-up Q&A ------------------------------------------
qa = requests.post(f"{VSS}/chat/completions", json={
    "id": file_id, "model": model,
    "messages": [{"role": "user", "content": "What is the color of the work suit?"}],
}, timeout=600).json()
print(qa["choices"][0]["message"]["content"])

# --- 5. Cleanup ----------------------------------------------------------
requests.delete(f"{VSS}/files/{file_id}")

KeyboardInterrupt: 